# 电影圈数据

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import scipy
import matplotlib.pyplot as plt
from networkx.algorithms import bipartite

In [2]:
import sys
sys.path.append("..")

# 数据处理

## 原始数据

In [3]:
df_raw = pd.read_csv("dwd_cast_works.csv")
df_raw.head()

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,k_genres,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray
0,1007816,凌玲,女,10767181,演员,是,5,10767181,妙手千金,电影,...,喜剧/动作,无评分,NaN,暂无评分,NaN,NaN,21534411,2015681,1,NaN
1,1003352,金焰,男,10618303,演员,是,1,10618303,失去的爱情,电影,...,爱情,无评分,NaN,暂无评分,NaN,NaN,21236655,2006753,2,NaN
2,1003352,金焰,男,10601198,演员,是,1,10601198,人道,电影,...,剧情,无评分,NaN,暂无评分,NaN,NaN,21202445,2006753,3,NaN
3,1003352,金焰,男,10566664,演员,是,2,10566664,母亲,电影,...,剧情,无评分,NaN,暂无评分,NaN,NaN,21133377,2006753,4,老邓
4,1013885,阿美莉嘉·奥利沃,女,10727641,演员,是,21,10727641,碟中谍5：神秘国度,电影,...,动作/惊悚/冒险,有评分,7.8,292193,292193.0,1510.0,21455331,2027819,5,Turandot


## 数据筛选
过滤条件
1. 电影
2. 有评分
3. 地区含中国

In [4]:
df_movie = df_raw.loc[
    (df_raw['k_type'] == '电影') &
    (df_raw['k_region'].str.contains('中国'))].copy()
# 仅保留演员，导演数
df_movie = df_movie[df_movie['k_role'].isin(['演员', '导演'])]
df_movie = df_movie.reset_index(drop=True).copy(deep=True)
# 添加新的movie_id_m列
df_movie['movie_id_m'] = df_movie['k_movie_id'].apply(lambda x: 'm' + str(x))
df_movie

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m
0,1007816,凌玲,女,10767181,演员,是,5,10767181,妙手千金,电影,...,无评分,NaN,暂无评分,NaN,NaN,21534411,2015681,1,NaN,m21534411
1,1003352,金焰,男,10618303,演员,是,1,10618303,失去的爱情,电影,...,无评分,NaN,暂无评分,NaN,NaN,21236655,2006753,2,NaN,m21236655
2,1003352,金焰,男,10601198,演员,是,1,10601198,人道,电影,...,无评分,NaN,暂无评分,NaN,NaN,21202445,2006753,3,NaN,m21202445
3,1003352,金焰,男,10566664,演员,是,2,10566664,母亲,电影,...,无评分,NaN,暂无评分,NaN,NaN,21133377,2006753,4,老邓,m21133377
4,1003424,蒋君超,男,10777680,演员,是,2,10777680,影坛风月,电影,...,无评分,NaN,暂无评分,NaN,NaN,21555409,2006897,6,NaN,m21555409
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224720,1370230,张巨光,男,3542526,演员,否,999,3542526,保卫胜利果实,电影,...,无评分,NaN,暂无评分,NaN,NaN,7085101,2740509,604126,村民,m7085101
224721,1431712,邓祥,男,1466295,演员,否,999,1466295,倚天屠龙记（上集）,电影,...,无评分,NaN,暂无评分,NaN,NaN,2932639,2863473,604127,史镖头,m2932639
224722,27496534,刘芊蒂,女,2337597,演员,是,2,2337597,操行零分,电影,...,有评分,7.0,211,211.0,38.0,4675243,54993117,604128,NaN,m4675243
224723,27565771,董亚春,女,2270516,导演,是,2,2270516,八月一日,电影,...,有评分,6.4,1123,1123.0,85.0,4541081,55131591,604130,NaN,m4541081


## 影人职责合并

In [5]:
# 合并影人职责
df_cast = df_movie[['k_cast_id', 'cast_name', 'k_role']].drop_duplicates()
df_cast_agg = df_cast.groupby(['k_cast_id', 'cast_name'])['k_role'].apply(lambda x: '/'.join(sorted(x.unique()))).reset_index()
df_cast_agg

,k_cast_id,cast_name,k_role
0,2000437,李香凝,演员
1,2000457,吉娜·卡拉诺,演员
2,2000515,理查德·格里克,演员
3,2000941,司汗,演员
4,2001011,凯文·格劳特,导演
...,...,...,...
42248,75617721,吴铃山,演员
42249,75617969,特蕾沙,演员
42250,75618493,何宥辰,演员
42251,75622879,祖丽米热,演员


In [6]:
df_movie['cast_role_agg'] = df_movie['k_cast_id'].map(
    df_cast_agg.set_index('k_cast_id')['k_role']
)
df_movie

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
0,1007816,凌玲,女,10767181,演员,是,5,10767181,妙手千金,电影,...,NaN,暂无评分,NaN,NaN,21534411,2015681,1,NaN,m21534411,演员
1,1003352,金焰,男,10618303,演员,是,1,10618303,失去的爱情,电影,...,NaN,暂无评分,NaN,NaN,21236655,2006753,2,NaN,m21236655,演员
2,1003352,金焰,男,10601198,演员,是,1,10601198,人道,电影,...,NaN,暂无评分,NaN,NaN,21202445,2006753,3,NaN,m21202445,演员
3,1003352,金焰,男,10566664,演员,是,2,10566664,母亲,电影,...,NaN,暂无评分,NaN,NaN,21133377,2006753,4,老邓,m21133377,演员
4,1003424,蒋君超,男,10777680,演员,是,2,10777680,影坛风月,电影,...,NaN,暂无评分,NaN,NaN,21555409,2006897,6,NaN,m21555409,导演/演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224720,1370230,张巨光,男,3542526,演员,否,999,3542526,保卫胜利果实,电影,...,NaN,暂无评分,NaN,NaN,7085101,2740509,604126,村民,m7085101,演员
224721,1431712,邓祥,男,1466295,演员,否,999,1466295,倚天屠龙记（上集）,电影,...,NaN,暂无评分,NaN,NaN,2932639,2863473,604127,史镖头,m2932639,演员
224722,27496534,刘芊蒂,女,2337597,演员,是,2,2337597,操行零分,电影,...,7.0,211,211.0,38.0,4675243,54993117,604128,NaN,m4675243,演员
224723,27565771,董亚春,女,2270516,导演,是,2,2270516,八月一日,电影,...,6.4,1123,1123.0,85.0,4541081,55131591,604130,NaN,m4541081,导演


In [ ]:
df_movie[df_movie['cast_name'] == '张艺谋']

## 有评分数据

In [7]:
df_movie_rated = df_movie[df_movie['is_rating'] == '有评分'].copy()
df_movie_rated = df_movie_rated.reset_index(drop=True).copy(deep=True)
df_movie_rated

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
0,1017182,陈燕燕,女,10771202,演员,是,1,10771202,深闺疑云,电影,...,7.8,92,92.0,27.0,21542453,2034413,54,赵兰,m21542453,演员
1,1043845,刘志荣,男,10581318,演员,是,11,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2087739,75,NaN,m21162685,导演/演员
2,1050373,茅瑛,女,10573512,演员,是,1,10573512,鬼娘子,电影,...,5.8,235,235.0,37.0,21147073,2100795,77,NaN,m21147073,演员
3,1124360,利智,女,10581318,演员,是,3,10581318,四大家族之龙虎兄弟,电影,...,7.8,2290,2290.0,134.0,21162685,2248769,92,NaN,m21162685,演员
4,1315281,卢庆辉,男,10490155,演员,是,1,10490155,衰鬼抓狂,电影,...,5.8,121,121.0,26.0,20980359,2630611,156,NaN,m20980359,演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100882,27541325,张世,男,1298203,演员,是,2,1298203,国道封闭,电影,...,7.9,146,146.0,34.0,2596455,55082699,604105,NaN,m2596455,导演/演员
100883,27517568,林迪安,男,1300498,演员,否,999,1300498,百变星君,电影,...,7.7,246162,246162.0,1377.0,2601045,55035185,604118,穿斑马服君,m2601045,演员
100884,27541395,韩鹏翼,男,24695588,演员,是,2,24695588,逆袭,电影,...,5.1,674,674.0,59.0,49391225,55082839,604119,NaN,m49391225,演员
100885,27496534,刘芊蒂,女,2337597,演员,是,2,2337597,操行零分,电影,...,7.0,211,211.0,38.0,4675243,54993117,604128,NaN,m4675243,演员


## 保存为csv

In [8]:
df_movie.to_csv("movie_data.csv", index=False)
df_movie_rated.to_csv("movie_data_rated.csv", index=False)

# 数据统计

## 年度数据

In [ ]:
# 按年统计每年的电影数量
df_count_by_year = df_movie.groupby('k_movie_year')['movie_id_m'].nunique().reset_index()
df_count_by_year.columns = ['year', 'movie_count']
df_count_by_year = df_count_by_year.sort_values('year')
df_count_by_year

In [ ]:
# 绘图
plt.figure(figsize=(10, 6))
plt.plot(df_count_by_year['year'], df_count_by_year['movie_count'], marker='o')
plt.title('年度电影数量统计')
plt.xlabel('年份')
plt.ylabel('电影数量')
plt.xticks(df_count_by_year['year'], rotation=45)
# plt.tight_layout()
plt.show()

# 图对象

In [ ]:
G = nx.from_pandas_edgelist(
    df_movie,
    source='k_cast_id',
    target='movie_id_m',
    edge_attr=True,
    create_using=nx.Graph()
)
G.number_of_nodes(), G.number_of_edges()

In [ ]:
# 子图数量
len(list(nx.connected_components(G)))

In [ ]:
# 最大联通分支
largest_cc = max(nx.connected_components(G), key=len)
G_largest = G.subgraph(largest_cc).copy()
G_largest.number_of_nodes(), G_largest.number_of_edges()

In [ ]:
# 删除度为1的节点
G_pruned = G_largest.copy()
nodes_to_remove = [node for node, degree in dict(G_pruned.degree()).items() if degree == 1]
G_pruned.remove_nodes_from(nodes_to_remove)
G_pruned.number_of_nodes(), G_pruned.number_of_edges()    

## 二分图

In [ ]:
# 二分图
nx.is_bipartite(G_pruned)

## 绘图

In [ ]:
def draw_network_largest_component(G):
    """绘制网络图"""
    # pos = nx.spring_layout(G, seed=42)
    # largest connected component
    components = nx.connected_components(G)
    largest_component = max(components, key=len)
    H = G.subgraph(largest_component)
    # compute centrality
    centrality = nx.betweenness_centrality(H, k=10, endpoints=True)
    # compute community structure
    lpc = nx.community.label_propagation_communities(H)
    community_index = {n: i for i, com in enumerate(lpc) for n in com}
    #### draw graph ####
    fig, ax = plt.subplots(figsize=(20, 20))
    pos = nx.spring_layout(H, k=0.15, seed=4572321)
    node_color = [community_index[n] for n in H]
    node_size = [v * 20000 for v in centrality.values()]
    return nx.draw_networkx(
        H,
        pos=pos,
        with_labels=False,
        node_color=node_color,
        node_size=node_size,
        edge_color="gainsboro",
        alpha=0.4,
    )

In [ ]:
draw_network_largest_component(G_pruned)

In [ ]:
def draw_network(G):
    """绘制网络图"""
    lpc = nx.community.label_propagation_communities(G)
    community_index = {n: i for i, com in enumerate(lpc) for n in com}
    #### draw graph ####
    fig, ax = plt.subplots(figsize=(20, 20))
    pos = nx.spring_layout(G, k=0.15, seed=4572321)
    node_color = [community_index[n] for n in G]
    return nx.draw_networkx(
        G,
        pos=pos,
        with_labels=False,
        node_color=node_color,
        edge_color="gainsboro",
        node_size=1,
        alpha=0.4,
    )

In [ ]:
draw_network(G_pruned)

# 图特征

## 定军山

In [ ]:
df_movie[df_movie['k_title'] == '定军山']

In [ ]:
root_id = 'm3830729'

In [ ]:
root_id in  G_pruned.nodes

## 图片测试

In [ ]:
df_2024 = df_movie_rated[df_movie_rated['k_movie_year'] == 2024].copy()
G_2024 = nx.from_pandas_edgelist(
    df_2024,
    source='k_cast_id',
    target='movie_id_m',
    edge_attr=True,
    create_using=nx.Graph()
)
G_2024.number_of_nodes(), G_2024.number_of_edges()

In [ ]:
pos = nx.spring_layout(G_2024, seed=42)
nx.draw(G_2024, pos, with_labels=False, node_size=1, node_color='skyblue', edge_color='gray', alpha=0.7)

In [ ]:
pos

In [ ]:
# 对pos的x坐标都乘以2
for key in pos:
    pos[key][0] *= 2
nx.draw(G_2024, pos, with_labels=False, node_size=1, node_color='skyblue', edge_color='gray', alpha=0.7)

In [ ]:
pos

In [ ]:
pos = nx.spring_layout(G_2024, seed=42)

plt.figure(figsize=(8,4))

# 第一次绘图
nx.draw(G_2024, pos, node_size=2, edge_color='gray', alpha=0.6)
plt.title("原始布局")
plt.axis('equal')
plt.show()

# 第二次绘图：x 坐标乘2
for k in pos:
    pos[k][0] *= 2

plt.figure(figsize=(8,4))
nx.draw(G_2024, pos, node_size=2, edge_color='gray', alpha=0.6)
plt.title("x方向放大2倍")
plt.axis('equal')

# 固定坐标范围，使得 matplotlib 不再自动缩放
x_vals, y_vals = zip(*pos.values())
plt.xlim(min(x_vals)-0.1, max(x_vals)+0.1)
plt.ylim(min(y_vals)-0.1, max(y_vals)+0.1)

plt.show()

# 年代数据

In [ ]:
df_movie_1945 = df_movie[df_movie['k_movie_year'] <= 1945].copy()
df_movie_1945

In [ ]:
movie_num_1945 = df_movie_1945['movie_id_m'].nunique()
movie_num_1945

In [ ]:
G_1945 = nx.from_pandas_edgelist(
    df_movie_1945,
    source='k_cast_id',
    target='movie_id_m',
    edge_attr=True,
    create_using=nx.Graph()     
)
G_1945.number_of_nodes(), G_1945.number_of_edges()

In [ ]:
# 二分
nx.is_bipartite(G_1945)


In [ ]:
cast_ids_1945 = df_movie_1945['k_cast_id'].unique().tolist()

In [ ]:
G_1945_bip = bipartite.projected_graph(G_1945, G_1945.nodes)

In [ ]:
G_1945_movie = G_1945_bip.copy()
G_1945_movie.remove_nodes_from(cast_ids_1945)

In [ ]:
G_1945_movie.number_of_nodes()

In [ ]:
pos = nx.spring_layout(G_1945_movie, seed=42)
for k in pos:
    pos[k][1] = df_movie_1945[df_movie_1945['movie_id_m'] == k]['k_movie_year'].values[0]
    pos[k][0] *= 1945
nx.draw(G_1945_movie, pos, node_size=2, edge_color='gray', alpha=0.6)

In [ ]:
df_movie_1945['k_movie_year'].min(), df_movie_1945['k_movie_year'].max()

In [ ]:
df_movie_1945[df_movie_1945['k_movie_year'] == 1905]